# 08 - Random and Magnitude-Based Baseline Pruning

**Project:** Regime-Conditional Attention Head Selection for Time Series Transformers (ReCAHS)

The project proposal calls for comparing the proposed regime-aware head selection method against four baselines: **no pruning**, **random head pruning**, **global importance pruning**, and **magnitude-based pruning**. Notebooks 04-07 already cover no pruning and global (validation-loss-based) importance pruning. This notebook fills in the two remaining baselines using the same B4 checkpoint, `HeadMaskController`, and evaluation routines as notebook 04, so all methods are directly comparable at the same 25% pruning ratio (6 of 24 heads):

1. **Random head pruning** — 6 of the 24 heads are disabled at random. A single random draw can be lucky or unlucky, so the experiment is repeated over 5 seeds (0-4) and reported as mean ± standard deviation.
2. **Magnitude-based pruning** — each head's importance is approximated by the L2 norm of its `query_projection` / `key_projection` / `value_projection` weight rows and its `out_projection` weight columns (a standard weight-magnitude heuristic, in contrast to the loss-based importance used for the "global importance" baseline). The 6 lowest-norm heads are pruned.

Both settings are evaluated on the validation set (with the same STL-based regime breakdown used elsewhere in the project) and on the held-out test set.


## 1. Mount Google Drive and import core libraries

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import os
import shutil
import random
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm.auto import tqdm

## 2. Define project paths

In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

REGIME_DIR = PROJECT_DIR / "regime_detection"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
HEAD_IMPORTANCE_DIR = PROJECT_DIR / "head_importance"
PRUNING_DIR = PROJECT_DIR / "pruning_experiments"

PRUNING_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("REGIME_DIR:", REGIME_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("HEAD_IMPORTANCE_DIR:", HEAD_IMPORTANCE_DIR)
print("PRUNING_DIR:", PRUNING_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
REGIME_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection
CHECKPOINT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints
HEAD_IMPORTANCE_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/head_importance
PRUNING_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments


## 3. Set up Time-Series-Library

If a fresh Colab runtime was started, the repository is cloned again. The ETTh1 dataset is copied from Drive if available, otherwise downloaded from the public ETDataset repository.

In [4]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))
print("Python path[0]:", sys.path[0])

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 19.47 MiB/s, done.
Resolving deltas: 100% (1570/1570), done.
Python path[0]: /content/Time-Series-Library


In [5]:
# TSLib importları için minimal paketler
!pip install -q patool sktime scikit-base --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 17.3 MB/s eta 0:00:00


In [6]:
drive_data_path = PROJECT_DIR / "data" / "ETTh1.csv"

tslib_data_path = (
    TSLIB_DIR
    / "dataset/ETDataset/ETT-small/ETTh1.csv"
)

tslib_data_path.parent.mkdir(parents=True, exist_ok=True)

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTh1 copied from Drive.")
else:
    print("Drive data not found. Downloading ETTh1...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv

print("Dataset exists:", tslib_data_path.exists())
print("Dataset path:", tslib_data_path)

df = pd.read_csv(tslib_data_path)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Drive data not found. Downloading ETTh1...
Dataset exists: True
Dataset path: /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv
Dataset shape: (17420, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


## 4. Load the regime labels

STL-based regime labels (trend / seasonal / residual) for every validation window, produced by `regime_detection.ipynb`. Used to break down pruning results by regime, matching the reporting format of notebooks 04-07.

In [7]:
regime_path = REGIME_DIR / "etth1_validation_regimes_ot_seq336.csv"
regime_df = pd.read_csv(regime_path)

print("Regime df shape:", regime_df.shape)
display(regime_df.head())
display(regime_df["regime"].value_counts())

Regime df shape: (2785, 13)


,window_id,start_idx,end_idx,input_start_date,input_end_date,trend_score,seasonal_score,residual_score,regime,top_score,second_score,confidence_margin,is_confident
0,0,0,336,2017-06-12 00:00:00,2017-06-25 23:00:00,0.700476,0.205640,0.093884,trend,0.700476,0.205640,0.494836,True
1,1,1,337,2017-06-12 01:00:00,2017-06-26 00:00:00,0.700166,0.206039,0.093796,trend,0.700166,0.206039,0.494127,True
2,2,2,338,2017-06-12 02:00:00,2017-06-26 01:00:00,0.696768,0.208689,0.094543,trend,0.696768,0.208689,0.488079,True
3,3,3,339,2017-06-12 03:00:00,2017-06-26 02:00:00,0.689886,0.213555,0.096559,trend,0.689886,0.213555,0.476330,True
4,4,4,340,2017-06-12 04:00:00,2017-06-26 03:00:00,0.677356,0.223502,0.099142,trend,0.677356,0.223502,0.453854,True


,count
regime,
trend,2134
residual,359
seasonal,292


## 5. Locate the B4 checkpoint

In [8]:
b4_checkpoint_candidates = list(
    CHECKPOINT_DIR.glob(
        "B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth"
    )
)

print("Found checkpoint candidates:")
for path in b4_checkpoint_candidates:
    print(path)

if len(b4_checkpoint_candidates) == 0:
    raise FileNotFoundError(
        "B4 checkpoint bulunamadı. CHECKPOINT_DIR içini kontrol et."
    )

b4_checkpoint_path = b4_checkpoint_candidates[0]
print("\nSelected checkpoint:")
print(b4_checkpoint_path)

Found checkpoint candidates:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth

Selected checkpoint:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth


## 6. Reconstruct the B4 model arguments

In [9]:
from argparse import Namespace

args = Namespace(
    # task
    task_name="long_term_forecast",
    is_training=0,
    model_id="ETTh1_336_96_dm128_h8",
    model="PatchTST",

    # data
    data="ETTh1",
    root_path="./dataset/ETDataset/ETT-small/",
    data_path="ETTh1.csv",
    features="M",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",

    # forecasting
    seq_len=336,
    label_len=48,
    pred_len=96,
    seasonal_patterns="Monthly",
    inverse=False,

    # model
    enc_in=7,
    dec_in=7,
    c_out=7,
    d_model=128,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=256,
    moving_avg=25,
    factor=3,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,

    # PatchTST related
    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    # optimization / loader
    num_workers=0,
    itr=1,
    train_epochs=10,
    batch_size=32,
    patience=3,
    learning_rate=0.0001,
    des="baseline_b4",
    loss="MSE",
    lradj="type1",
    use_amp=False,

    # GPU
    use_gpu=torch.cuda.is_available(),
    gpu=0,
    use_multi_gpu=False,
    devices="0",
    gpu_type="cuda",
    # other model families, required by some imports
    expand=2,
    d_conv=4,
    top_k=5,
    num_kernels=6,
    channel_independence=0,
    decomp_method="moving_avg",
    use_norm=1,
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    seg_len=48,

    # MLP projection args sometimes expected
    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
)

print(args)

Namespace(task_name='long_term_forecast', is_training=0, model_id='ETTh1_336_96_dm128_h8', model='PatchTST', data='ETTh1', root_path='./dataset/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=336, label_len=48, pred_len=96, seasonal_patterns='Monthly', inverse=False, enc_in=7, dec_in=7, c_out=7, d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=3, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, patch_len=16, stride=8, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, num_workers=0, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='baseline_b4', loss='MSE', lradj='type1', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0', gpu_type='cuda', expand=2, d_conv=4, top_k=5, num_kernels=6, channel_independence=0, decomp_method='moving_avg', use_norm=1, down_s

## 7. Load the model and checkpoint weights

In [10]:
%cd /content/Time-Series-Library

/content/Time-Series-Library


In [11]:
!pip install reformer-pytorch --no-deps
!pip install local-attention --no-deps
!pip install hyper_connections --no-deps
!pip install axial_positional_embedding --no-deps
!pip install product_key_memory --no-deps
!pip install colt5_attention --no-deps

In [12]:
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

exp = Exp_Long_Term_Forecast(args)
model = exp.model.to(device)

checkpoint = torch.load(
    b4_checkpoint_path,
    map_location=device,
)

model.load_state_dict(checkpoint)
model.eval()

print("Model loaded successfully.")

Device: cuda:0
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
Model loaded successfully.


## 8. Build the validation and test loaders

In [13]:
from data_provider.data_factory import data_provider

vali_data, vali_loader = data_provider(
    args,
    flag="val",
)

test_data, test_loader = data_provider(
    args,
    flag="test",
)

print("Validation dataset length:", len(vali_data))
print("Validation loader batches:", len(vali_loader))
print("Test dataset length:", len(test_data))
print("Test loader batches:", len(test_loader))

assert len(vali_data) == len(regime_df), (
    len(vali_data),
    len(regime_df),
)

print("Validation windows and regime labels match.")

val 2785
test 2785
Validation dataset length: 2785
Validation loader batches: 88
Test dataset length: 2785
Test loader batches: 88
Validation windows and regime labels match.


## 9. HeadMaskController

Monkey-patches each encoder attention layer's forward pass so that an arbitrary subset of heads can be zeroed out at inference time, without modifying the underlying PatchTST implementation. Identical to the controller used in notebook 04.

In [14]:
class HeadMaskController:
    def __init__(self, model):
        self.model = model
        self.original_forwards = {}
        self.current_mask = None

    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention

            if layer_idx in self.original_forwards:
                continue

            original_forward = attention_layer.forward
            self.original_forwards[layer_idx] = original_forward

            def make_masked_forward(layer_idx, attention_layer):
                def masked_forward(
                    queries,
                    keys,
                    values,
                    attn_mask,
                    tau=None,
                    delta=None,
                ):
                    B, L, _ = queries.shape
                    _, S, _ = keys.shape
                    H = attention_layer.n_heads

                    queries_proj = attention_layer.query_projection(queries)
                    keys_proj = attention_layer.key_projection(keys)
                    values_proj = attention_layer.value_projection(values)

                    queries_proj = queries_proj.view(B, L, H, -1)
                    keys_proj = keys_proj.view(B, S, H, -1)
                    values_proj = values_proj.view(B, S, H, -1)

                    out, attn = attention_layer.inner_attention(
                        queries_proj,
                        keys_proj,
                        values_proj,
                        attn_mask,
                        tau=tau,
                        delta=delta,
                    )

                    if self.current_mask is not None:
                        layer_mask = self.current_mask[layer_idx].to(out.device)
                        layer_mask = layer_mask.view(1, 1, H, 1)
                        out = out * layer_mask

                    out = out.view(B, L, -1)

                    return attention_layer.out_projection(out), attn

                return masked_forward

            attention_layer.forward = make_masked_forward(
                layer_idx,
                attention_layer,
            )

    def remove(self):
        for layer_idx, original_forward in self.original_forwards.items():
            self.model.encoder.attn_layers[layer_idx].attention.forward = original_forward

        self.original_forwards = {}
        self.current_mask = None

    def set_all_active(self):
        num_layers = len(self.model.encoder.attn_layers)
        num_heads = self.model.encoder.attn_layers[0].attention.n_heads

        self.current_mask = torch.ones(
            num_layers,
            num_heads,
            dtype=torch.float32,
        )

    def set_mask(self, mask):
        self.current_mask = mask.clone().float()

    def build_mask_from_prune_list(self, prune_pairs):
        """prune_pairs: iterable of (layer_idx, head_idx) to switch off."""
        self.set_all_active()

        for layer_idx, head_idx in prune_pairs:
            self.current_mask[layer_idx, head_idx] = 0.0

        return self.current_mask.clone()


In [15]:
mask_controller = HeadMaskController(model)
mask_controller.install()

num_layers = len(model.encoder.attn_layers)
num_heads = model.encoder.attn_layers[0].attention.n_heads
total_heads = num_layers * num_heads
n_prune = round(total_heads * 0.25)

print("Layers:", num_layers)
print("Heads per layer:", num_heads)
print("Total heads:", total_heads)
print("Heads to prune (25%):", n_prune)

baseline_mask = torch.ones(num_layers, num_heads)

Layers: 3
Heads per layer: 8
Total heads: 24
Heads to prune (25%): 6


## 10. Loss computation helpers

`compute_regime_losses_with_mask` evaluates a given head mask on the validation set and reports both the overall MSE/MAE and a per-regime breakdown. `compute_test_metrics_with_mask` evaluates a mask on the test set (no regime breakdown, since regime labels are only used for validation-side analysis here).

In [16]:
mse_criterion = nn.MSELoss(reduction="none")
mae_criterion = nn.L1Loss(reduction="none")

def compute_regime_losses_with_mask(
    model,
    loader,
    regime_df,
    mask_controller,
    mask,
    device,
    pred_len=96,
    desc="validation",
):
    model.eval()
    mask_controller.set_mask(mask)

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark,
            )

            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            batch_size = batch_x.shape[0]

            for i in range(batch_size):
                window_id = global_index + i
                regime = regime_df.iloc[window_id]["regime"]

                all_records.append({
                    "window_id": window_id,
                    "regime": regime,
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })

            global_index += batch_size

    result_df = pd.DataFrame(all_records)

    regime_summary = (
        result_df
        .groupby("regime")
        .agg(
            mse=("mse", "mean"),
            mae=("mae", "mean"),
            count=("window_id", "count"),
        )
        .reset_index()
    )

    overall = {
        "overall_mse": float(result_df["mse"].mean()),
        "overall_mae": float(result_df["mae"].mean()),
    }

    return {
        "overall": overall,
        "regime_summary": regime_summary,
        "window_losses": result_df,
    }


def compute_test_metrics_with_mask(
    model,
    loader,
    mask_controller,
    mask,
    device,
    pred_len=96,
    desc="test",
):
    model.eval()
    mask_controller.set_mask(mask)

    all_mse = []
    all_mae = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark,
            )

            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            all_mse.extend(mse_per_sample.detach().cpu().numpy().tolist())
            all_mae.extend(mae_per_sample.detach().cpu().numpy().tolist())

    return {
        "test_mse": float(np.mean(all_mse)),
        "test_mae": float(np.mean(all_mae)),
    }


## 11. B4 baseline (no pruning) reference values

Computed once, with all 24 heads active, and reused as the comparison point for both the random and magnitude baselines below.

In [17]:
baseline_val = compute_regime_losses_with_mask(
    model=model,
    loader=vali_loader,
    regime_df=regime_df,
    mask_controller=mask_controller,
    mask=baseline_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Baseline validation",
)

baseline_test = compute_test_metrics_with_mask(
    model=model,
    loader=test_loader,
    mask_controller=mask_controller,
    mask=baseline_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Baseline test",
)

print("Baseline validation overall:", baseline_val["overall"])
print("Baseline test:", baseline_test)
display(baseline_val["regime_summary"])

Baseline validation:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline test:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline validation overall: {'overall_mse': 0.678066410600497, 'overall_mae': 0.5550830315216654}
Baseline test: {'test_mse': 0.37254557146739276, 'test_mae': 0.3982142798241422}


,regime,mse,mae,count
0,residual,0.700110,0.562844,359
1,seasonal,0.663310,0.555660,292
2,trend,0.676377,0.553698,2134


## 12. Random head pruning (5 seeds)

For each seed, 6 of 24 heads are selected uniformly at random and disabled. This is the **random head pruning** baseline requested in the project proposal — averaging over multiple seeds avoids reporting a result that is only good (or bad) by chance.

In [18]:
RANDOM_SEEDS = [0, 1, 2, 3, 4]

def build_random_prune_pairs(num_layers, num_heads, n_prune, seed):
    rng = random.Random(seed)
    all_pairs = [
        (layer_idx, head_idx)
        for layer_idx in range(num_layers)
        for head_idx in range(num_heads)
    ]
    return rng.sample(all_pairs, n_prune)


random_val_rows = []
random_test_rows = []
random_regime_rows = []
random_masks = {}
random_prune_lists = {}

for seed in RANDOM_SEEDS:
    prune_pairs = build_random_prune_pairs(num_layers, num_heads, n_prune, seed)
    mask = mask_controller.build_mask_from_prune_list(prune_pairs)

    random_masks[seed] = mask.clone()
    random_prune_lists[seed] = prune_pairs

    val_result = compute_regime_losses_with_mask(
        model=model,
        loader=vali_loader,
        regime_df=regime_df,
        mask_controller=mask_controller,
        mask=mask,
        device=device,
        pred_len=args.pred_len,
        desc=f"Random pruning seed={seed} validation",
    )

    test_result = compute_test_metrics_with_mask(
        model=model,
        loader=test_loader,
        mask_controller=mask_controller,
        mask=mask,
        device=device,
        pred_len=args.pred_len,
        desc=f"Random pruning seed={seed} test",
    )

    random_val_rows.append({
        "seed": seed,
        "validation_mse": val_result["overall"]["overall_mse"],
        "validation_mae": val_result["overall"]["overall_mae"],
    })

    random_test_rows.append({
        "seed": seed,
        "test_mse": test_result["test_mse"],
        "test_mae": test_result["test_mae"],
    })

    regime_seed_df = val_result["regime_summary"].copy()
    regime_seed_df["seed"] = seed
    random_regime_rows.append(regime_seed_df)

random_val_df = pd.DataFrame(random_val_rows)
random_test_df = pd.DataFrame(random_test_rows)
random_regime_df = pd.concat(random_regime_rows, ignore_index=True)

print("Per-seed validation results:")
display(random_val_df)

print("Per-seed test results:")
display(random_test_df)

Random pruning seed=0 validation:   0%|          | 0/88 [00:00<?, ?it/s]

Random pruning seed=0 test:   0%|          | 0/88 [00:00<?, ?it/s]

Random pruning seed=1 validation:   0%|          | 0/88 [00:00<?, ?it/s]

Random pruning seed=1 test:   0%|          | 0/88 [00:00<?, ?it/s]

Random pruning seed=2 validation:   0%|          | 0/88 [00:00<?, ?it/s]

...

Random pruning seed=4 test:   0%|          | 0/88 [00:00<?, ?it/s]

Per-seed validation results:


,seed,validation_mse,validation_mae
0,0,0.675099,0.556347
1,1,0.700978,0.566630
2,2,0.685726,0.561420
3,3,0.703117,0.565138
4,4,0.668398,0.552788


Per-seed test results:


,seed,test_mse,test_mae
0,0,0.377759,0.402349
1,1,0.377283,0.401048
2,2,0.381037,0.405017
3,3,0.382536,0.403353
4,4,0.380782,0.401388


In [19]:
random_val_summary = {
    "setting": "B4_random_pruning_25",
    "pruned_heads": n_prune,
    "active_heads": total_heads - n_prune,
    "pruning_ratio": n_prune / total_heads,
    "validation_mse_mean": random_val_df["validation_mse"].mean(),
    "validation_mse_std": random_val_df["validation_mse"].std(),
    "validation_mae_mean": random_val_df["validation_mae"].mean(),
    "validation_mae_std": random_val_df["validation_mae"].std(),
}

random_test_summary = {
    "setting": "B4_random_pruning_25",
    "pruned_heads": n_prune,
    "active_heads": total_heads - n_prune,
    "pruning_ratio": n_prune / total_heads,
    "test_mse_mean": random_test_df["test_mse"].mean(),
    "test_mse_std": random_test_df["test_mse"].std(),
    "test_mae_mean": random_test_df["test_mae"].mean(),
    "test_mae_std": random_test_df["test_mae"].std(),
}

print("Random pruning validation summary (mean over 5 seeds):")
print(random_val_summary)

print("\nRandom pruning test summary (mean over 5 seeds):")
print(random_test_summary)

print("\nBaseline validation:", baseline_val["overall"])
print("Baseline test:", baseline_test)

print(
    "\nRandom pruning changed validation MSE by "
    f"{(random_val_summary['validation_mse_mean'] - baseline_val['overall']['overall_mse']):.6f} "
    f"({(random_val_summary['validation_mse_mean'] / baseline_val['overall']['overall_mse'] - 1) * 100:.3f}%)."
)

print(
    "Random pruning changed test MSE by "
    f"{(random_test_summary['test_mse_mean'] - baseline_test['test_mse']):.6f} "
    f"({(random_test_summary['test_mse_mean'] / baseline_test['test_mse'] - 1) * 100:.3f}%)."
)

Random pruning validation summary (mean over 5 seeds):
{'setting': 'B4_random_pruning_25', 'pruned_heads': 6, 'active_heads': 18, 'pruning_ratio': 0.25, 'validation_mse_mean': np.float64(0.6866635325117642), 'validation_mse_std': 0.015361410812153392, 'validation_mae_mean': np.float64(0.5604646434240323), 'validation_mae_std': 0.0058448551257302694}

Random pruning test summary (mean over 5 seeds):
{'setting': 'B4_random_pruning_25', 'pruned_heads': 6, 'active_heads': 18, 'pruning_ratio': 0.25, 'test_mse_mean': np.float64(0.3798793071594016), 'test_mse_std': 0.002261031012300298, 'test_mae_mean': np.float64(0.40263114257282584), 'test_mae_std': 0.0016081346004244784}

Baseline validation: {'overall_mse': 0.678066410600497, 'overall_mae': 0.5550830315216654}
Baseline test: {'test_mse': 0.37254557146739276, 'test_mae': 0.3982142798241422}

Random pruning changed validation MSE by 0.008597 (1.268%).
Random pruning changed test MSE by 0.007334 (1.969%).


## 13. Magnitude-based head pruning

Head importance is approximated from weight magnitude rather than validation loss (see Michel et al., 2019 — proposal reference [7] — for the broader literature on head-pruning criteria). For each head `h`:

- `query_projection.weight[h*d_k:(h+1)*d_k, :]`
- `key_projection.weight[h*d_k:(h+1)*d_k, :]`
- `value_projection.weight[h*d_v:(h+1)*d_v, :]`
- `out_projection.weight[:, h*d_v:(h+1)*d_v]`

are concatenated and their Frobenius (L2) norm is computed. The 6 heads with the smallest norm are treated as least influential and pruned. This is the **magnitude-based pruning** baseline requested in the project proposal.

In [20]:
def compute_head_magnitudes(model):
    records = []

    for layer_idx, encoder_layer in enumerate(model.encoder.attn_layers):
        attention_layer = encoder_layer.attention
        H = attention_layer.n_heads

        d_k = attention_layer.query_projection.out_features // H
        d_v = attention_layer.value_projection.out_features // H

        q_w = attention_layer.query_projection.weight.detach()
        k_w = attention_layer.key_projection.weight.detach()
        v_w = attention_layer.value_projection.weight.detach()
        o_w = attention_layer.out_projection.weight.detach()

        for head_idx in range(H):
            q_slice = q_w[head_idx * d_k:(head_idx + 1) * d_k, :]
            k_slice = k_w[head_idx * d_k:(head_idx + 1) * d_k, :]
            v_slice = v_w[head_idx * d_v:(head_idx + 1) * d_v, :]
            o_slice = o_w[:, head_idx * d_v:(head_idx + 1) * d_v]

            magnitude = torch.sqrt(
                q_slice.pow(2).sum()
                + k_slice.pow(2).sum()
                + v_slice.pow(2).sum()
                + o_slice.pow(2).sum()
            ).item()

            records.append({
                "layer": layer_idx,
                "head": head_idx,
                "magnitude": magnitude,
            })

    return pd.DataFrame(records)


head_magnitude_df = compute_head_magnitudes(model)
head_magnitude_df = head_magnitude_df.sort_values("magnitude").reset_index(drop=True)

print("Heads sorted by magnitude (ascending, lowest = pruning candidates):")
display(head_magnitude_df)

Heads sorted by magnitude (ascending, lowest = pruning candidates):


,layer,head,magnitude
0,1,2,4.636657
1,2,4,4.650136
2,1,3,4.660092
3,2,3,4.664751
4,2,5,4.668289
5,1,5,4.677190
6,0,0,4.680248
7,2,7,4.683477
8,1,7,4.683867
9,2,0,4.685277


In [21]:
magnitude_prune_df = head_magnitude_df.head(n_prune).copy()

print("Magnitude-based pruning candidates (lowest weight norm):")
display(magnitude_prune_df[["layer", "head", "magnitude"]])

assert len(magnitude_prune_df) == n_prune

magnitude_prune_pairs = list(
    zip(magnitude_prune_df["layer"], magnitude_prune_df["head"])
)

magnitude_prune_mask = mask_controller.build_mask_from_prune_list(magnitude_prune_pairs)

print("\nMagnitude prune mask:")
print(magnitude_prune_mask)

Magnitude-based pruning candidates (lowest weight norm):


,layer,head,magnitude
0,1,2,4.636657
1,2,4,4.650136
2,1,3,4.660092
3,2,3,4.664751
4,2,5,4.668289
5,1,5,4.677190



Magnitude prune mask:
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 0., 0., 1., 0., 1., 1.],
        [1., 1., 1., 0., 0., 0., 1., 1.]])


In [22]:
magnitude_val = compute_regime_losses_with_mask(
    model=model,
    loader=vali_loader,
    regime_df=regime_df,
    mask_controller=mask_controller,
    mask=magnitude_prune_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Magnitude pruning validation",
)

magnitude_test = compute_test_metrics_with_mask(
    model=model,
    loader=test_loader,
    mask_controller=mask_controller,
    mask=magnitude_prune_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Magnitude pruning test",
)

print("Magnitude-pruned validation overall:", magnitude_val["overall"])
print("Magnitude-pruned test:", magnitude_test)
display(magnitude_val["regime_summary"])

print(
    "\nMagnitude pruning changed validation MSE by "
    f"{(magnitude_val['overall']['overall_mse'] - baseline_val['overall']['overall_mse']):.6f} "
    f"({(magnitude_val['overall']['overall_mse'] / baseline_val['overall']['overall_mse'] - 1) * 100:.3f}%)."
)

print(
    "Magnitude pruning changed test MSE by "
    f"{(magnitude_test['test_mse'] - baseline_test['test_mse']):.6f} "
    f"({(magnitude_test['test_mse'] / baseline_test['test_mse'] - 1) * 100:.3f}%)."
)

Magnitude pruning validation:   0%|          | 0/88 [00:00<?, ?it/s]

Magnitude pruning test:   0%|          | 0/88 [00:00<?, ?it/s]

Magnitude-pruned validation overall: {'overall_mse': 0.6683907765808182, 'overall_mae': 0.5522495830615505}
Magnitude-pruned test: {'test_mse': 0.38097486596026153, 'test_mae': 0.4043945164933025}


,regime,mse,mae,count
0,residual,0.681806,0.555656,359
1,seasonal,0.594668,0.531221,292
2,trend,0.676221,0.554554,2134



Magnitude pruning changed validation MSE by -0.009676 (-1.427%).
Magnitude pruning changed test MSE by 0.008429 (2.263%).


## 14. Save results

Outputs are written under `pruning_experiments/b4_random_pruning_25/` and `pruning_experiments/b4_magnitude_pruning_25/` on Drive, following the same directory convention as notebook 04's `b4_static_pruning_25/`.

In [23]:
random_dir = PRUNING_DIR / "b4_random_pruning_25"
random_dir.mkdir(parents=True, exist_ok=True)

random_val_df.to_csv(random_dir / "per_seed_validation_results.csv", index=False)
random_test_df.to_csv(random_dir / "per_seed_test_results.csv", index=False)
random_regime_df.to_csv(random_dir / "per_seed_regime_validation_results.csv", index=False)

pd.DataFrame([random_val_summary]).to_csv(random_dir / "validation_summary.csv", index=False)
pd.DataFrame([random_test_summary]).to_csv(random_dir / "test_summary.csv", index=False)

for seed, pairs in random_prune_lists.items():
    pd.DataFrame(pairs, columns=["layer", "head"]).to_csv(
        random_dir / f"pruned_heads_seed{seed}.csv", index=False
    )

print("Saved random pruning outputs to:", random_dir)
for path in sorted(random_dir.iterdir()):
    print(" -", path.name)

Saved random pruning outputs to: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_random_pruning_25
 - per_seed_regime_validation_results.csv
 - per_seed_test_results.csv
 - per_seed_validation_results.csv
 - pruned_heads_seed0.csv
 - pruned_heads_seed1.csv
 - pruned_heads_seed2.csv
 - pruned_heads_seed3.csv
 - pruned_heads_seed4.csv
 - test_summary.csv
 - validation_summary.csv


In [24]:
magnitude_dir = PRUNING_DIR / "b4_magnitude_pruning_25"
magnitude_dir.mkdir(parents=True, exist_ok=True)

head_magnitude_df.to_csv(magnitude_dir / "head_magnitude_ranking.csv", index=False)
magnitude_prune_df.to_csv(magnitude_dir / "pruned_heads.csv", index=False)

mask_df = pd.DataFrame(
    magnitude_prune_mask.numpy(),
    index=[f"layer_{i}" for i in range(magnitude_prune_mask.shape[0])],
    columns=[f"head_{j}" for j in range(magnitude_prune_mask.shape[1])],
)
mask_df.to_csv(magnitude_dir / "magnitude_prune_mask.csv")

val_comparison = pd.DataFrame([
    {
        "setting": "B4_no_pruning",
        "validation_mse": baseline_val["overall"]["overall_mse"],
        "validation_mae": baseline_val["overall"]["overall_mae"],
    },
    {
        "setting": "B4_magnitude_pruning_25",
        "validation_mse": magnitude_val["overall"]["overall_mse"],
        "validation_mae": magnitude_val["overall"]["overall_mae"],
    },
])

test_comparison = pd.DataFrame([
    {
        "setting": "B4_no_pruning",
        "test_mse": baseline_test["test_mse"],
        "test_mae": baseline_test["test_mae"],
    },
    {
        "setting": "B4_magnitude_pruning_25",
        "test_mse": magnitude_test["test_mse"],
        "test_mae": magnitude_test["test_mae"],
    },
])

val_comparison.to_csv(magnitude_dir / "validation_overall_comparison.csv", index=False)
test_comparison.to_csv(magnitude_dir / "test_overall_comparison.csv", index=False)
magnitude_val["regime_summary"].to_csv(magnitude_dir / "validation_regime_summary.csv", index=False)

print("Saved magnitude pruning outputs to:", magnitude_dir)
for path in sorted(magnitude_dir.iterdir()):
    print(" -", path.name)

Saved magnitude pruning outputs to: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_magnitude_pruning_25
 - head_magnitude_ranking.csv
 - magnitude_prune_mask.csv
 - pruned_heads.csv
 - test_overall_comparison.csv
 - validation_overall_comparison.csv
 - validation_regime_summary.csv


## 15. Combined summary table

Consolidates the no-pruning baseline with both new baselines into a single table. Use these values to fill in the "Experiment 08" section of `README.md` and the corresponding rows in `results/experiment_summary.csv`.

In [25]:
summary_rows = [
    {
        "experiment_id": "B4_no_pruning",
        "method": "no_pruning",
        "pruning_ratio": 0.0,
        "val_mse": baseline_val["overall"]["overall_mse"],
        "val_mae": baseline_val["overall"]["overall_mae"],
        "test_mse": baseline_test["test_mse"],
        "test_mae": baseline_test["test_mae"],
    },
    {
        "experiment_id": "B4_random_pruning_25",
        "method": "random_pruning_5seed_mean",
        "pruning_ratio": n_prune / total_heads,
        "val_mse": random_val_summary["validation_mse_mean"],
        "val_mae": random_val_summary["validation_mae_mean"],
        "test_mse": random_test_summary["test_mse_mean"],
        "test_mae": random_test_summary["test_mae_mean"],
    },
    {
        "experiment_id": "B4_magnitude_pruning_25",
        "method": "magnitude_based_pruning",
        "pruning_ratio": n_prune / total_heads,
        "val_mse": magnitude_val["overall"]["overall_mse"],
        "val_mae": magnitude_val["overall"]["overall_mae"],
        "test_mse": magnitude_test["test_mse"],
        "test_mae": magnitude_test["test_mae"],
    },
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(PRUNING_DIR / "experiment_08_random_magnitude_summary.csv", index=False)
print("Saved:", PRUNING_DIR / "experiment_08_random_magnitude_summary.csv")

,experiment_id,method,pruning_ratio,val_mse,val_mae,test_mse,test_mae
0,B4_no_pruning,no_pruning,0.00,0.678066,0.555083,0.372546,0.398214
1,B4_random_pruning_25,random_pruning_5seed_mean,0.25,0.686664,0.560465,0.379879,0.402631
2,B4_magnitude_pruning_25,magnitude_based_pruning,0.25,0.668391,0.552250,0.380975,0.404395


Saved: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/experiment_08_random_magnitude_summary.csv
